In [2]:
import feedparser

NEWS_SOURCES = {
    "BBC": "http://feeds.bbci.co.uk/news/rss.xml",
    "CNN": "http://rss.cnn.com/rss/edition.rss",
    "Al Jazeera": "https://www.aljazeera.com/xml/rss/all.xml",
    "CNBC": "https://www.cnbc.com/id/100003114/device/rss/rss.html"
}

def fetch_news(topic=None, max_articles=10):
    articles = []

    for source, url in NEWS_SOURCES.items():
        feed = feedparser.parse(url)
        for entry in feed.entries:
            title = entry.title
            summary = entry.get("summary", "")
            link = entry.link
            article = {"source": source, "title": title, "summary": summary, "link": link}

            if topic:
                if topic.lower() in title.lower() or topic.lower() in summary.lower():
                    articles.append(article)
            else:
                articles.append(article)

            if len(articles) >= max_articles:
                return articles
    return articles

def combine_articles(articles):
    text = ""
    for article in articles:
        text += article["title"] + "\n"
        text += article["summary"] + "\n\n"
    return text


In [3]:
from news_fetcher import fetch_news
articles = fetch_news(topic="AI", max_articles=5)
for a in articles:
    print(a["title"], a["source"])


Four crew members killed after US refuelling plane crashes in Iraq BBC
Woman found out she had terminal brain cancer after suitcase fell on her head BBC
Mission accomplished? The 2003 boast that haunts today's Iran conflict BBC
AI toys for children misread emotions and respond inappropriately, researchers warn BBC
Britons should not take photos of strikes in UAE, embassy warns BBC


In [4]:
import requests

def generate_news(source_text, topic, word_limit):
    prompt = f"""
You are a professional journalist.
Using the information below, write a news article about: {topic}

Requirements:
- Professional news style
- No repetition
- Maximum {word_limit} words

Sources:
{source_text}
"""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": prompt, "stream": False}
    )
    return response.json()["response"]


In [ ]:
from ai_generator import generate_news
text = "AI is transforming healthcare. New models detect diseases faster."
article = generate_news(text, "AI in Healthcare", 150)
print(article)


In [10]:
# publisher
import os
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")


def post_to_discord(message: str) -> bool:
    """
    Sends a message to Discord using a webhook.

    Returns True if successful, otherwise False.
    """

    if not DISCORD_WEBHOOK_URL:
        raise ValueError("DISCORD_WEBHOOK_URL not found in .env file")

    payload = {
        "content": message[:1900]  # Discord message limit protection
    }

    try:
        response = requests.post(DISCORD_WEBHOOK_URL, json=payload, timeout=10)

        if response.status_code in (200, 204):
            return True
        else:
            print("Discord error:", response.text)
            return False

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        return False


In [ ]:
import streamlit as st
from news_fetcher import fetch_news, combine_articles
from ai_generator import generate_news
from publisher import post_to_discord

st.set_page_config(page_title="AI News Generator", layout="centered")

st.title("AI News Generator → Discord Publisher")

topic = st.text_input("Enter News Topic", "Artificial Intelligence")
word_limit = st.slider("Article Word Limit", 50, 500, 150)

generated_article = None

if st.button("Generate News"):

    with st.spinner("Fetching latest news..."):
        articles = fetch_news(topic, max_articles=5)

    if not articles:
        st.warning("No articles found.")
    else:
        source_text = combine_articles(articles)

        with st.spinner("Generating AI article..."):
            generated_article = generate_news(source_text, topic, word_limit)

        st.subheader("Generated Article")
        st.write(generated_article)

        st.session_state["article"] = generated_article


if "article" in st.session_state:

    if st.button("Publish to Discord"):

        success = post_to_discord(st.session_state["article"])

        if success:
            st.success("Article successfully published to Discord!")
        else:
            st.error("Failed to publish article.")
